In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [ ]:
"""
H3 Analysis: License Family and Violation Subtype
-----------------------------------------------------
H3. Procedural incompatibilities are disproportionately associated
with permissive rather than reciprocal origin licenses.

Design notes:
- License family is classified by exact dict lookup through DSR_engine.py (license_mapping -> get_license_group). H1a uses this identical 
  implementation, so the two hypotheses' license-family classifications cannot drift apart.
- base_repository_url is not used here. H3 operates on violation category and source-side license family, not project identity.

Scope restriction (structural, not incidental):
Every Category 4 (structural/high-risk) row in this dataset carries an unresolved origin license, consistent with the LCD taxonomy's
definition of Category 4 as covering unknown-origin ingestion as well as structural incompatibility. License-known-ness is therefore part of
the category's own assignment criteria, and testing license family as a predictor of Procedural vs. Structural across both categories would
be circular. The comparative test is restricted to WITHIN Category 3, where category assignment does not depend on the origin's license
family.

This script therefore:
  1. Reports the Category 4 / unresolved-license pattern as a taxonomy observation, not a modeled association.
  2. Tests Permissive vs. Reciprocal shares within Category 3 by one-sample chi-square goodness-of-fit against an equal baseline,
     with Cohen's h as the effect size.

Reproduces: Category 3 analyzed N = 66,136 (33,773 Permissive / 32,363 Reciprocal, 51.07% / 48.93%); chi2(1) = 30.06, p = 4.19e-08;
Cohen's h = 0.043 (negligible). Category 4: n = 17,147, 0.00% with a resolved origin license.
"""

import numpy as np
import pandas as pd
from scipy.stats import chisquare, chi2_contingency

FILE_PATH = DATA / "license_analysis_results_processed.csv"
TARGET_LANGS = ["C", "C++", "C#", "Java", "JavaScript", "Python"]

# ---- License classification: DSR_engine.py is the single source of truth ----
_ns = {}
exec(open(DATA / "DSR_engine.py").read(), _ns)  # adjust path if DSR_engine.py lives elsewhere
license_mapping = _ns["license_mapping"]
get_license_group = _ns["get_license_group"]


def classify_license(lic):
    """Returns 'Permissive', 'Reciprocal', or 'Unresolved' (never a
    silent NaN, so nothing gets dropped without accounting).

    Exact dict lookup: raw license string -> SPDX id (via
    license_mapping) -> compatibility group (via get_license_group)
    -> Permissive / Weak+Strong Copyleft (folded into 'Reciprocal') a
    nything else (Custom, Restricted, Proprietary, Unknown,Unlicensed -> 'Unresolved'). 
    No substring matching, so no risk of false-positive keyword collisions (e.g. "upl" matching inside
    "eupl"). Raw strings not present in license_mapping at all also resolve to Unresolved 
    -- a deliberate conservative default, not a silent failure. Identical to the classify_license() used in H1a.
    """
    if pd.isna(lic):
        return "Unresolved"
    spdx = license_mapping.get(str(lic).strip(), None)
    if spdx is None:
        return "Unresolved"
    group = get_license_group(spdx)
    if group == "Permissive":
        return "Permissive"
    if group in ("Weak Copyleft", "Strong Copyleft"):
        return "Reciprocal"
    return "Unresolved"


def cramers_v_corrected(contingency_table):
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    r, c = contingency_table.shape
    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - ((c - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    v = np.sqrt(phi2_corr / min(c_corr - 1, r_corr - 1))
    return v, chi2, p, dof


def cohens_h(p1, p2):
    """Cohen's h for two proportions.

    h = | 2*arcsin(sqrt(p1))  -  2*arcsin(sqrt(p2)) |

    NOTE the factor of 2 on EACH arcsine term. Omitting it yields roughly
    half the correct value (for H3: 0.021 instead of 0.043) and was the
    source of an error in an earlier draft. Conventional thresholds:
    0.2 small, 0.5 medium, 0.8 large.
    """
    phi1 = 2.0 * np.arcsin(np.sqrt(p1))
    phi2 = 2.0 * np.arcsin(np.sqrt(p2))
    return abs(phi1 - phi2)


def evaluate_h3():
    df = pd.read_csv(FILE_PATH)
    df = df.replace(r"^\s*$", np.nan, regex=True)

    before = len(df)
    df = df.drop_duplicates(subset=["method_hash", "source_repository_url", "sink_repository_url"])
    print(f"[dedup] {before} -> {len(df)} rows ({before - len(df)} duplicates removed)")

    df["language_clean"] = df["language"].replace({"JS": "JavaScript"})
    df = df[df["language_clean"].isin(TARGET_LANGS)].copy()

    df["license_family"] = df["source_file_license"].apply(classify_license)

    # =========================================================================
    # [1] Diagnostic: Category 3 vs Category 4 raw composition (no dropping)
    # =========================================================================
    df_actionable = df[df["violation_lcd_category"].isin([3, 4])].copy()
    print(f"\nTotal actionable observations (Cat 3 & 4, target languages): {len(df_actionable):,}")
    print(f"  Category 3 (Procedural): {(df_actionable['violation_lcd_category'] == 3).sum():,}")
    print(f"  Category 4 (Structural): {(df_actionable['violation_lcd_category'] == 4).sum():,}")

    print("\n=== Diagnostic: license_family composition by category ===")
    diag = pd.crosstab(df_actionable["violation_lcd_category"], df_actionable["license_family"])
    print(diag)

    cat4_resolved_share = 1 - (
        (df_actionable.loc[df_actionable["violation_lcd_category"] == 4, "license_family"] == "Unresolved").mean()
    )
    print(f"\n[diagnostic] Share of Category 4 rows with a RESOLVED (named) license: "
          f"{cat4_resolved_share:.2%}")
    if cat4_resolved_share == 0:
        print("[!] Category 4 is 100% associated with unresolved license metadata in this "
              "dataset. This is consistent with the LCD taxonomy's definition of Category 4 "
              "as covering unknown-origin ingestion, and precludes testing license family as "
              "an independent predictor of violation subtype without circularity. Reported "
              "here as a taxonomy observation, not modeled further below.")

    # =========================================================================
    # [2] Substantive test: Permissive vs. Reciprocal share, WITHIN Category 3 only
    #     (non-circular: Category 3 assignment does not depend on license family)
    # =========================================================================
    df_cat3 = df_actionable[df_actionable["violation_lcd_category"] == 3].copy()
    df_cat3_named = df_cat3[df_cat3["license_family"].isin(["Permissive", "Reciprocal"])].copy()

    n_excluded_unresolved = len(df_cat3) - len(df_cat3_named)
    print(f"\n=== Category 3 (Procedural) license family distribution ===")
    print(f"Total Category 3 observations: {len(df_cat3):,}")
    print(f"  Excluded (Unresolved license): {n_excluded_unresolved:,} "
          f"({n_excluded_unresolved/len(df_cat3):.2%})")
    print(f"  Analyzed (Permissive or Reciprocal): {len(df_cat3_named):,}")

    counts = df_cat3_named["license_family"].value_counts()
    print("\n" + counts.to_string())
    print((counts / counts.sum() * 100).round(2).astype(str) + "%")

    expected = [counts.sum() / 2, counts.sum() / 2]
    chi2_stat, p_val = chisquare(counts.values, f_exp=expected)
    print(f"\nOne-sample Chi-Square Goodness-of-Fit (Permissive vs. Reciprocal, "
          f"H0: equal distribution):")
    print(f"  chi2(1) = {chi2_stat:.2f}, p = {p_val:.4g}")

    # Effect size. chi2 grows with N, so the verdict rests on this, not on p.
    n_total = counts.sum()
    p_perm = counts.get("Permissive", 0) / n_total
    p_recip = counts.get("Reciprocal", 0) / n_total
    h = cohens_h(p_perm, p_recip)
    magnitude = ("negligible" if h < 0.2 else "small" if h < 0.5
                 else "medium" if h < 0.8 else "large")
    print(f"  p1 (Permissive) = {p_perm:.6f}   p2 (Reciprocal) = {p_recip:.6f}")
    print(f"  Cohen's h = {h:.4f} ({magnitude})")

    # =========================================================================
    # [3] Summary for manuscript
    # =========================================================================
    print("\n" + "=" * 60)
    print("SUMMARY FOR H3 WRITE-UP")
    print("=" * 60)
    print(f"1. Category 4 is exclusively associated with unresolved origin license "
          f"({cat4_resolved_share:.0%} resolved), consistent with the taxonomy's own "
          f"definition -- reported as a taxonomy observation, not an independent test.")
    print(f"2. Within Category 3, {counts.get('Permissive', 0):,} "
          f"({counts.get('Permissive', 0)/counts.sum():.2%}) originate from permissive "
          f"licenses vs. {counts.get('Reciprocal', 0):,} "
          f"({counts.get('Reciprocal', 0)/counts.sum():.2%}) from reciprocal licenses "
          f"(chi2={chi2_stat:.2f}, p={p_val:.4g}, Cohen's h={h:.4f} -- negligible).")
    print(f"3. The original comparative form of H3 (license family predicting violation "
          f"subtype across BOTH Categories 3 and 4) could not be tested due to structural "
          f"confounding between license-known-ness and category assignment.")

    return {
        "diagnostic_table": diag,
        "cat4_resolved_share": cat4_resolved_share,
        "cat3_counts": counts,
        "chi2": chi2_stat,
        "p_value": p_val,
        "cohens_h": h,
    }


if __name__ == "__main__":
    results = evaluate_h3()

[dedup] 1183182 -> 1183182 rows (0 duplicates removed)

Total actionable observations (Cat 3 & 4, target languages): 138,556
  Category 3 (Procedural): 121,409
  Category 4 (Structural): 17,147

=== Diagnostic: license_family composition by category ===
license_family          Permissive  Reciprocal  Unresolved
violation_lcd_category                                    
3                            33773       32363       55273
4                                0           0       17147

[diagnostic] Share of Category 4 rows with a RESOLVED (named) license: 0.00%
[!] Category 4 is 100% associated with unresolved license metadata in this dataset. This is consistent with the LCD taxonomy's definition of Category 4 as covering unknown-origin ingestion, and precludes testing license family as an independent predictor of violation subtype without circularity. Reported here as a taxonomy observation, not modeled further below.

=== Category 3 (Procedural) license family distribution ===
Total 